In [1]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"  # 禁止联网，只用本地 cache

# 策略读取

In [2]:
import torch
# Swap this import per-policy
from lerobot.policies.pi05 import PI05Policy

# load a policy
model_id = "/vla/.models/lerobot-pi05_base"  # <- swap checkpoint
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy = PI05Policy.from_pretrained(model_id).to(device).eval()

/mnt/workspace/luyi/.cache/miniconda3/envs/lerobot_pi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/workspace/luyi/.cache/miniconda3/envs/lerobot_pi/lib/python3.10/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


The PI05 model is a direct port of the OpenPI implementation. 
This implementation follows the original OpenPI structure for compatibility. 
Original implementation: https://github.com/Physical-Intelligence/openpi


Loading model from: /vla/.models/lerobot-pi05_base


✓ Loaded state dict from model.safetensors
Remapped: action_in_proj.bias -> model.action_in_proj.bias
Remapped: action_in_proj.weight -> model.action_in_proj.weight
Remapped: action_out_proj.bias -> model.action_out_proj.bias
Remapped: action_out_proj.weight -> model.action_out_proj.weight
Remapped: paligemma_with_expert.gemma_expert.lm_head.weight -> model.paligemma_with_expert.gemma_expert.lm_head.weight
Remapped: paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.bias -> model.paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.bias
Remapped: paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.weight -> model.paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.weight
Remapped: paligemma_with_expert.gemma_expert.model.layers.0.mlp.down_proj.weight -> model.paligemma_with_expert.gemma_expert.model.layers.0.mlp.down_proj.weight
Remapped: paligemma_with_expert.gemma_expert.model.layers.0.mlp.gate_proj.weigh

In [3]:
policy.config

PI05Config(n_obs_steps=1, input_features={'observation.images.robot0_agentview_left_image': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 224, 224)), 'observation.images.robot0_eye_in_hand_image': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 224, 224)), 'observation.images.robot0_agentview_right_image': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 224, 224)), 'observation.state': PolicyFeature(type=<FeatureType.STATE: 'STATE'>, shape=(32,))}, output_features={'action': PolicyFeature(type=<FeatureType.ACTION: 'ACTION'>, shape=(32,))}, device='cuda', use_amp=True, use_peft=False, push_to_hub=False, repo_id=None, private=None, tags=None, license=None, pretrained_path=None, paligemma_variant='gemma_2b', action_expert_variant='gemma_300m', dtype='bfloat16', chunk_size=1, n_action_steps=1, max_state_dim=32, max_action_dim=32, num_inference_steps=10, time_sampling_beta_alpha=1.5, time_sampling_beta_beta=1.0, time_sampling_scale=0.999, time_sampli

In [4]:
policy.config.input_features

{'observation.images.robot0_agentview_left_image': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 224, 224)),
 'observation.images.robot0_eye_in_hand_image': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 224, 224)),
 'observation.images.robot0_agentview_right_image': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 224, 224)),
 'observation.state': PolicyFeature(type=<FeatureType.STATE: 'STATE'>, shape=(32,))}

In [5]:
policy.config.output_features

{'action': PolicyFeature(type=<FeatureType.ACTION: 'ACTION'>, shape=(32,))}

### 数据集

In [6]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset
dataset_path = "/vla/.data/test"
ds = LeRobotDataset(repo_id=dataset_path)
ds

LeRobotDataset({
    Repository ID: '/vla/.data/test',
    Number of selected episodes: '1',
    Number of selected samples: '436',
    Features: '['observation.state', 'action', 'observation.images.robot0_agentview_left_image', 'observation.images.robot0_agentview_right_image', 'observation.images.robot0_eye_in_hand_image', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']',
})',

In [7]:
ds.meta.info

{'codebase_version': 'v3.0',
 'robot_type': 'franka_panda',
 'total_episodes': 1,
 'total_frames': 436,
 'total_tasks': 1,
 'chunks_size': 1000,
 'data_files_size_in_mb': 100,
 'video_files_size_in_mb': 200,
 'fps': 20,
 'splits': {'train': '0:1'},
 'data_path': 'data/chunk-{chunk_index:03d}/file-{file_index:03d}.parquet',
 'video_path': 'videos/{video_key}/chunk-{chunk_index:03d}/file-{file_index:03d}.mp4',
 'features': {'observation.state': {'dtype': 'float64',
   'shape': (9,),
   'names': ['gripper_qpos_left',
    'gripper_qpos_right',
    'eef_pos_x',
    'eef_pos_y',
    'eef_pos_z',
    'eef_quat_w',
    'eef_quat_x',
    'eef_quat_y',
    'eef_quat_z']},
  'action': {'dtype': 'float64',
   'shape': (12,),
   'names': ['rel_pos_x',
    'rel_pos_y',
    'rel_pos_z',
    'rel_rot_6d_0',
    'rel_rot_6d_1',
    'rel_rot_6d_2',
    'rel_rot_6d_3',
    'rel_rot_6d_4',
    'rel_rot_6d_5',
    'gripper',
    'extra_0',
    'extra_1']},
  'observation.images.robot0_agentview_left_image'

In [8]:
ds[1]

{'observation.images.robot0_agentview_left_image': tensor([[[0.7137, 0.7137, 0.7137,  ..., 0.5137, 0.5137, 0.5137],
          [0.7137, 0.7137, 0.7137,  ..., 0.5137, 0.5137, 0.5137],
          [0.7137, 0.7137, 0.7137,  ..., 0.5137, 0.5137, 0.5137],
          ...,
          [0.1216, 0.1216, 0.1216,  ..., 0.5647, 0.5765, 0.5804],
          [0.1216, 0.1216, 0.1216,  ..., 0.6078, 0.6275, 0.6275],
          [0.1216, 0.1216, 0.1216,  ..., 0.6078, 0.6275, 0.6275]],
 
         [[0.6980, 0.6980, 0.6980,  ..., 0.5137, 0.5137, 0.5137],
          [0.6980, 0.6980, 0.6980,  ..., 0.5137, 0.5137, 0.5137],
          [0.6980, 0.6980, 0.6980,  ..., 0.5137, 0.5137, 0.5137],
          ...,
          [0.1216, 0.1216, 0.1216,  ..., 0.4510, 0.4471, 0.4510],
          [0.1216, 0.1216, 0.1216,  ..., 0.4549, 0.4510, 0.4510],
          [0.1216, 0.1216, 0.1216,  ..., 0.4549, 0.4510, 0.4510]],
 
         [[0.6745, 0.6745, 0.6745,  ..., 0.5137, 0.5137, 0.5137],
          [0.6745, 0.6745, 0.6745,  ..., 0.5137, 0.5137,

数据处理管线

In [9]:
from lerobot.policies.factory import make_pre_post_processors
preprocess, postprocess = make_pre_post_processors(
    policy_cfg=policy.config,
    pretrained_path=model_id,
)
preprocess.__dict__['steps']

[RenameObservationsProcessorStep(rename_map={}),
 AddBatchDimensionProcessorStep(to_batch_action_processor=AddBatchDimensionActionStep(), to_batch_observation_processor=AddBatchDimensionObservationStep(), to_batch_complementary_data_processor=AddBatchDimensionComplementaryDataStep()),
 NormalizerProcessorStep(features={}, norm_map={<FeatureType.VISUAL: 'VISUAL'>: <NormalizationMode.IDENTITY: 'IDENTITY'>, <FeatureType.STATE: 'STATE'>: <NormalizationMode.QUANTILES: 'QUANTILES'>, <FeatureType.ACTION: 'ACTION'>: <NormalizationMode.QUANTILES: 'QUANTILES'>}, stats={}, device=None, dtype=torch.float32, eps=1e-08, normalize_observation_keys=None),
 Pi05PrepareStateTokenizerProcessorStep(max_state_dim=32, task_key='task'),
 TokenizerProcessorStep(tokenizer_name='google/paligemma-3b-pt-224', tokenizer=None, max_length=200, task_key='task', padding_side='right', padding='max_length', truncation=True),
 DeviceProcessorStep(device='cpu', float_dtype=None)]

In [10]:
print("预处理前数据keys",ds[0].keys())
batch = preprocess(ds[0])
batch.keys() # batch中是包括action的

预处理前数据keys dict_keys(['observation.images.robot0_agentview_left_image', 'observation.images.robot0_agentview_right_image', 'observation.images.robot0_eye_in_hand_image', 'observation.state', 'action', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index', 'task'])


dict_keys(['action', 'next.reward', 'next.done', 'next.truncated', 'info', 'task', 'index', 'task_index', 'episode_index', 'observation.images.robot0_agentview_left_image', 'observation.images.robot0_agentview_right_image', 'observation.images.robot0_eye_in_hand_image', 'observation.state', 'observation.language.tokens', 'observation.language.attention_mask'])

In [11]:
def to_device(batch):
    return {
        k: v.to(device, non_blocking=True) if isinstance(v, torch.Tensor) else v
        for k, v in batch.items()
    }
batch = to_device(batch)

In [12]:
# select_action —— 推理接口
with torch.inference_mode():
    pred_action_raw = policy.select_action(batch)
pred_action_raw

tensor([[-0.0306,  0.0181,  0.4735, -0.9683,  1.0231,  0.0180, -1.1170,  0.0209,
          0.2134, -0.0193,  0.0081, -0.0792, -0.0386,  0.0524,  0.2054,  0.4314,
          0.0414,  0.0210,  0.0761,  0.1998,  0.0337,  0.0480, -0.0300, -0.0265,
         -0.0076, -0.0041,  0.0171,  0.0091, -0.0061, -0.0134, -0.0164,  0.0163]],
       device='cuda:0')

In [13]:
# config.chunk_size 当前为1 训练时设定为50
with torch.inference_mode(): # 不进行反向传播
    loss, output_dict = policy.forward(batch)
print("loss:", loss.item() if hasattr(loss, "item") else loss)
print("output keys:", output_dict.keys() if isinstance(output_dict, dict) else type(output_dict))

loss: 0.4249928891658783
output keys: dict_keys(['loss_per_dim', 'loss'])


/mnt/workspace/luyi/lerobot/src/lerobot/policies/pi05/modeling_pi05.py:777: UserWarning: Using a target size (torch.Size([1, 1, 32])) that is different to the input size (torch.Size([1, 32])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(u_t, v_t, reduction="none")
